# Sector Rotation Strategy - Inference Walkthrough

This notebook demonstrates the step-by-step process of running inference with the RRG sector rotation strategy.

**Using OPTIMIZED parameters from Bayesian optimization (Sharpe = 0.650)**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import strategy components
from strategies import RRGStrategy, RRGConfig
from backtesting import Backtester, WalkForwardValidator, WalkForwardConfig

## Step 1: Load Data

In [ ]:
# Load the sector price data
df = pd.read_csv("data/processed/indian_sector_data_2013_2025.csv", parse_dates=["Date"])
df = df.set_index("Date").sort_index()
df = df.apply(pd.to_numeric, errors="coerce").ffill().dropna(how="all")

# Separate benchmark and sector prices
benchmark = df["Benchmark"]
prices = df.drop(columns=["Benchmark"])

print(f"Data range: {prices.index[0]} to {prices.index[-1]}")
print(f"Sectors: {list(prices.columns)}")
prices.head()

## Step 2: Configure and Fit Strategy

Using **OPTIMIZED** parameters from Bayesian optimization.

In [ ]:
# OPTIMIZED CONFIGURATION (from Bayesian optimization)
# Walk-Forward OOS Sharpe: 0.650
config = RRGConfig(
    top_n_sectors=5,           # Select top 5 sectors
    max_sector_weight=0.30,    # Max 30% per sector
    min_sector_weight=0.01,    # Min 1% per sector
    rs_lookback=100,           # RS-Ratio lookback (weeks)
    momentum_lookback=30,      # Momentum lookback (weeks)
    volatility_window=46,      # Vol window for weighting
    use_trend_filter=True,     # Only allocate if benchmark > 50-day MA
    trend_ma_period=50,        # Trend filter MA period
    weight_smoothing_alpha=0.3,# EMA smoothing for turnover reduction
    full_allocation=True,      # Always allocate 100%
    rebalance_frequency="W-FRI"  # Weekly on Friday
)

print(config)

In [ ]:
# Fit the strategy
strategy = RRGStrategy(config)
strategy.fit(prices, benchmark)

print(f"Strategy fitted: {strategy.is_fitted}")

## Step 3: Inspect RRG Signals

View the RS-Ratio and RS-Momentum for each sector.

In [ ]:
# Get RRG quadrant classification for today
latest_date = prices.index[-1]
quadrants = strategy.get_quadrant_classification(latest_date)

print(f"RRG Classification as of {latest_date.date()}\n")
display(quadrants.sort_values("RS_Momentum", ascending=False))

In [ ]:
# Visualize RRG quadrants
fig, ax = plt.subplots(figsize=(10, 8))

colors = {
    "Leading": "green",
    "Weakening": "orange",
    "Lagging": "red",
    "Improving": "blue"
}

for sector, row in quadrants.iterrows():
    ax.scatter(row["RS_Ratio"], row["RS_Momentum"], 
               color=colors.get(row["Quadrant"], "gray"), s=100)
    ax.annotate(sector, (row["RS_Ratio"], row["RS_Momentum"]), 
                fontsize=9, ha="left")

ax.axhline(100, color="black", linewidth=0.5, linestyle="--")
ax.axvline(100, color="black", linewidth=0.5, linestyle="--")
ax.set_xlabel("RS-Ratio")
ax.set_ylabel("RS-Momentum")
ax.set_title(f"RRG Quadrant Chart - {latest_date.date()}")
ax.text(101, 101, "Leading", fontsize=10, color="green")
ax.text(101, 99, "Weakening", fontsize=10, color="orange")
ax.text(99, 99, "Lagging", fontsize=10, color="red")
ax.text(99, 101, "Improving", fontsize=10, color="blue")
plt.tight_layout()
plt.show()

## Step 4: Predict Weights (Live Inference)

In [ ]:
# Get current portfolio weights
weights = strategy.predict_weights(prices, latest_date)

print(f"Portfolio Weights as of {latest_date.date()}")
print(f"Total Allocation: {weights.sum():.2%}\n")

active = weights[weights > 0].sort_values(ascending=False)
for sector, w in active.items():
    print(f"  {sector}: {w:.2%}")

## Step 5: Run Full Backtest

In [ ]:
# Create backtester and run
backtester = Backtester(strategy, risk_free_rate=0.05)
result = backtester.run(prices, benchmark)

# Print performance report
backtester.print_report(result)

In [ ]:
# Plot equity curves
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(result.strategy_equity, label="Strategy", linewidth=2)
ax.plot(result.benchmark_equity, label="Benchmark", linewidth=1, alpha=0.7)

ax.set_xlabel("Date")
ax.set_ylabel("Equity")
ax.set_title("Strategy vs Benchmark Equity Curve")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
# Plot drawdowns
def compute_drawdown(equity):
    return (equity / equity.cummax()) - 1

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(result.strategy_equity.index, 
                compute_drawdown(result.strategy_equity), 
                0, alpha=0.5, label="Strategy DD")
ax.fill_between(result.benchmark_equity.index, 
                compute_drawdown(result.benchmark_equity), 
                0, alpha=0.3, label="Benchmark DD")
ax.set_ylabel("Drawdown")
ax.set_title("Drawdown Comparison")
ax.legend()
plt.tight_layout()
plt.show()

## Step 6: Walk-Forward Validation

In [ ]:
# Configure walk-forward validation
wf_config = WalkForwardConfig(
    train_window=504,   # ~2 years
    test_window=63,     # ~3 months
    step_size=63        # Step forward by test window
)

# Run validation
strategy_fresh = RRGStrategy(config)  # Fresh instance
validator = WalkForwardValidator(strategy_fresh, config=wf_config)
wf_result = validator.validate(prices, benchmark)

validator.print_report(wf_result)

In [ ]:
# Plot OOS equity curve
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(wf_result.oos_equity, label="Out-of-Sample Equity", linewidth=2)
ax.set_xlabel("Date")
ax.set_ylabel("Equity")
ax.set_title("Walk-Forward Out-of-Sample Performance")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# View per-fold performance
wf_result.fold_metrics[["fold", "test_start", "test_end", "total_return", "max_drawdown"]]

---

## Summary (Optimized Strategy)

| Metric | Backtest | Walk-Forward (OOS) |
|--------|----------|--------------------|
| Total Return | 455% | 404% |
| CAGR | 14.3% | 14.8% |
| Sharpe | 0.612 | 0.650 |
| Max DD | -36.0% | -34.9% |
| Fold Profitability | — | 70.2% |